# AI-Based Spark Job Performance Optimization System

This notebook trains a machine-learning model on simulated PySpark job execution
metrics (record count, partitions, data skew, caching) to **predict job execution
time** and **recommend optimal Spark configurations** (executor memory, cores,
partitions, caching) for a given workload profile.

**Pipeline:** live PySpark micro-benchmarks → EDA → model comparison
(Linear / ElasticNet / RandomForest / GradientBoosting / XGBoost) → XGBoost
hyperparameter tuning → AI recommendation engine → default-vs-optimized comparison
→ results dashboard.

See `README.md` for setup, dataset, and results.


## 1. Setup

In [ ]:
!pip install pyspark scikit-learn pandas numpy matplotlib seaborn joblib -q

In [ ]:
import os, time, json, warnings, random,glob
from collections import defaultdict
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings("ignore")

from pyspark.sql            import SparkSession
from pyspark.sql.functions  import col, rand, when
from pyspark.sql.types      import StructType, StructField, IntegerType, DoubleType

from sklearn.ensemble        import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model    import LinearRegression, ElasticNet
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline        import Pipeline
from xgboost                 import XGBRegressor
import joblib

In [ ]:
random.seed(42)
np.random.seed(42)

## 2. Spark Session

In [ ]:
import os
import sys
from pyspark.sql import SparkSession

PYTHON_PATH = sys.executable
os.environ["PYSPARK_PYTHON"] = PYTHON_PATH
os.environ["PYSPARK_DRIVER_PYTHON"] = PYTHON_PATH
os.environ["SPARK_EVENTLOG_ENABLED"] = "false"
os.makedirs(os.path.join("tmp", "spark-events"), exist_ok=True)

In [ ]:
def create_spark_session(executor_memory="2g", cores=2, partitions=100):

    spark = (
        SparkSession.builder
        .appName("AI_Spark_Optimizer")
        .master("local[*]")
        .config("spark.executor.memory", executor_memory)
        .config("spark.executor.cores", str(cores))
        .config("spark.sql.shuffle.partitions", str(partitions))
        .config("spark.driver.memory", "2g")
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("ERROR")
    return spark


spark = create_spark_session()

print(spark.version)
print(spark.sparkContext.pythonExec)

## 3. Load Dataset

Place `Spark_realtime_metrices.csv` in the `data/` folder at the project root (see `data/README.md`).

In [ ]:
print("Retriveing Spark execution dataset …")
df_raw = pd.read_csv("../data/Spark_realtime_metrices.csv")
print(f"Dataset shape : {df_raw.shape}")
print(df_raw.describe().round(2))
df_raw

## 4. Live PySpark Benchmarks

In [ ]:
def run_spark_benchmark(spark, num_rows=500_000, n_partitions=100,
                        label="default"):
    """Execute a representative PySpark workload and measure wall-clock time."""
    spark.conf.set("spark.sql.shuffle.partitions", str(n_partitions))

    schema = StructType([
        StructField("id",    IntegerType(), True),
        StructField("value", DoubleType(),  True),
        StructField("group", IntegerType(), True),
    ])
    random.seed(42)
    np.random.seed(42)
    t0 = time.time()

    # --- generate data ---
    data = [(i, random.gauss(50, 15), i % 10) for i in range(num_rows)]
    rdd  = spark.sparkContext.parallelize(data, n_partitions)
    sdf  = spark.createDataFrame(rdd, schema)

    # --- transformations ---
    result = (
        sdf.filter(col("value") > 30)
           .groupBy("group")
           .agg({"value": "mean", "id": "count"})
           .orderBy("group")
    )
    result.collect()

    elapsed = round(time.time() - t0, 3)
    print(f"  [{label:20s}]  partitions={n_partitions:4d}  "
          f"rows={num_rows:>7,}  time={elapsed:.3f}s")
    return elapsed


print("\nRunning live Spark benchmarks …\n")
benchmark_results = []

configs_to_test = [
    (200_000, 10,  "small_data_few_parts"),
    (200_000, 100, "small_data_many_parts"),
    (500_000, 50,  "medium_balanced"),
    (500_000, 200, "medium_many_parts"),
    (500_000, 10,  "medium_few_parts"),
]

for rows, parts, lbl in configs_to_test:
    t = run_spark_benchmark(spark, num_rows=rows, n_partitions=parts, label=lbl)
    benchmark_results.append({"label": lbl, "rows": rows,
                               "partitions": parts, "time_sec": t})

df_bench = pd.DataFrame(benchmark_results)
print("\nBenchmark complete.\n")
print(df_bench.to_string(index=False))

## 5. Exploratory Data Analysis

In [ ]:
print("\nGenerating EDA plots …")

fig = plt.figure(figsize=(18, 12))
fig.suptitle("Spark Job Execution – Exploratory Data Analysis",
             fontsize=16, fontweight="bold", y=1.01)
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df_raw["execution_time_sec"], bins=40, color="#2196F3", edgecolor="white")
ax1.set_title("Distribution of Execution Time"); ax1.set_xlabel("Seconds")

ax2 = fig.add_subplot(gs[0, 1])
corr = df_raw.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            ax=ax2, linewidths=0.5)
ax2.set_title("Feature Correlation Matrix")

ax3 = fig.add_subplot(gs[1, 0])
sc = ax3.scatter(df_raw["partitions"], df_raw["execution_time_sec"],
                 c=df_raw["num_records"], cmap="viridis", alpha=0.5, s=15)
plt.colorbar(sc, ax=ax3, label="Num Records")
ax3.set_title("Partitions vs Exec Time"); ax3.set_xlabel("Partitions")

ax4 = fig.add_subplot(gs[1, 1])
ax4.scatter(df_raw["data_skew"], df_raw["execution_time_sec"],
            color="#E91E63", alpha=0.4, s=15)
ax4.set_title("Data Skew vs Exec Time"); ax4.set_xlabel("Skew Factor (0–1)")

plt.savefig("eda_plots.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Model Training & Comparison

In [ ]:
FEATURES = ["num_records", "partitions", "data_skew", "cache_enabled"]
TARGET   = "execution_time_sec"

X = df_raw[FEATURES].values
y = df_raw[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Linear Regression" : Pipeline([
        ("sc", StandardScaler()),
        ("m",  LinearRegression()),
    ]),
    "Elastic Net"        : Pipeline([
        ("sc", StandardScaler()),
        ("m",  ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42)),
    ]),
    "Random Forest"      : RandomForestRegressor(
        n_estimators=150, max_depth=12, random_state=42, n_jobs=-1),
    "Gradient Boosting"  : GradientBoostingRegressor(
        n_estimators=150, learning_rate=0.08, max_depth=5, random_state=42),
    "XGBoost"            : XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        objective="reg:squarederror", random_state=42, n_jobs=-1),
}
 
results_ml = {}
print("\n🤖  Training ML models …\n")
print(f"{'Model':<25} {'MAE':>8} {'RMSE':>8} {'R²':>8}  CV-R² (±std)")
print("─" * 65)
 
for name, model in models.items():
    model.fit(X_train, y_train)
    preds      = model.predict(X_test)
    mae        = mean_absolute_error(y_test, preds)
    rmse       = np.sqrt(mean_squared_error(y_test, preds))
    r2         = r2_score(y_test, preds)
    cv_scores  = cross_val_score(model, X, y, cv=5, scoring="r2")
    results_ml[name] = {"model": model, "MAE": mae, "RMSE": rmse,
                         "R2": r2, "CV_mean": cv_scores.mean(),
                         "CV_std": cv_scores.std()}
    print(f"{name:<25} {mae:>8.2f} {rmse:>8.2f} {r2:>8.4f}"
          f"  {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")
 
# Pick best model by CV R²
best_name  = max(results_ml, key=lambda k: results_ml[k]["CV_mean"])
best_model = results_ml[best_name]["model"]
print(f"\n🏆  Best model (initial) : {best_name}")
print(f"    CV R²               : {results_ml[best_name]['CV_mean']:.4f}")

## 7. XGBoost Hyperparameter Tuning

In [ ]:
print("\nTuning XGBoost hyperparameters …\n")
 
xgb_base = XGBRegressor(objective="reg:squarederror", random_state=42, n_jobs=-1)
 
param_grid = {
    "n_estimators"    : [100, 200, 300, 500],
    "max_depth"       : [3, 4, 5, 6, 8],
    "learning_rate"   : [0.01, 0.03, 0.05, 0.08, 0.1],
    "subsample"       : [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "gamma"           : [0, 0.1, 0.2, 0.3],
    "reg_alpha"       : [0, 0.01, 0.1, 0.5, 1],
    "reg_lambda"      : [0.5, 1, 1.5, 2, 5],
    "min_child_weight": [1, 3, 5, 7],
}
 
xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=40,            # raise to 80+ for a deeper search
    scoring="r2",
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1,
)
xgb_search.fit(X_train, y_train)
 
best_xgb   = xgb_search.best_estimator_
xgb_preds  = best_xgb.predict(X_test)
xgb_mae    = mean_absolute_error(y_test, xgb_preds)
xgb_rmse   = np.sqrt(mean_squared_error(y_test, xgb_preds))
xgb_r2     = r2_score(y_test, xgb_preds)
 
print("\nTuned XGBoost – Best Parameters:")
print(xgb_search.best_params_)
print(f"\nTuned XGBoost Performance")
print(f"  MAE  : {xgb_mae:.2f}")
print(f"  RMSE : {xgb_rmse:.2f}")
print(f"  R²   : {xgb_r2:.4f}")
print(f"  CV R²: {xgb_search.best_score_:.4f}")
 
results_ml["XGBoost (tuned)"] = {
    "model"   : best_xgb,
    "MAE"     : xgb_mae,
    "RMSE"    : xgb_rmse,
    "R2"      : xgb_r2,
    "CV_mean" : xgb_search.best_score_,
    "CV_std"  : 0.0,
}
 
best_name  = max(results_ml, key=lambda k: results_ml[k]["R2"])
best_model = results_ml[best_name]["model"]
print(f"\nFinal best model : {best_name}")
print(f"    CV R²           : {results_ml[best_name]['R2']:.4f}")

## 8. Feature Importance

In [ ]:
def get_feature_importances(model):
    """Extract feature importances from the best model or fall back to RF."""
    inner = model.named_steps["m"] if hasattr(model, "named_steps") else model
    if hasattr(inner, "feature_importances_"):
        return inner.feature_importances_, best_name
    # Linear / Elastic Net: use |coefficients| as proxy importance
    if hasattr(inner, "coef_"):
        coefs = np.abs(inner.coef_)
        return coefs / coefs.sum(), f"{best_name} |coef|"
    # Ultimate fallback — use RF (should rarely happen)
    rf_fallback = results_ml["Random Forest"]["model"]
    return rf_fallback.feature_importances_, "Random Forest (fallback)"
 
importances, fi_source = get_feature_importances(best_model)
fi_df = (pd.DataFrame({"Feature": FEATURES, "Importance": importances})
           .sort_values("Importance", ascending=True))
 
fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#1565C0" if v > 0.15 else "#42A5F5" for v in fi_df["Importance"]]
ax.barh(fi_df["Feature"], fi_df["Importance"], color=colors)
ax.set_title(f"Feature Importance – {fi_source}", fontsize=13, fontweight="bold")
ax.set_xlabel("Importance Score")
for i, (feat, val) in enumerate(zip(fi_df["Feature"], fi_df["Importance"])):
    ax.text(val + 0.002, i, f"{val:.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Candidate Configuration Grid

In [ ]:
CANDIDATE_CONFIGS = []
for mem   in [2, 4, 8, 16]:
    for cores in [1, 2, 4, 8]:
        for parts in [50, 100, 200, 400]:
            for cache in [0, 1]:
                CANDIDATE_CONFIGS.append({
                    "executor_memory_gb": mem,
                    "cores"             : cores,
                    "partitions"        : parts,
                    "cache_enabled"     : cache,
                })

## 10. Simulation & AI Recommendation Engine

In [ ]:
def simulate_spark_execution(
    num_records: int,
    partitions: int,
    executor_memory_gb: float,
    cores: int,
    data_skew: float,
    cache_enabled: bool,
):
    """
    Lightweight simulation of a Spark job.
    Returns (estimated_exec_sec, shuffle_size_mb).
 
    The exec_sec here is a physics-based rough estimate only;
    the trained ML model provides the authoritative prediction.
    shuffle_size_mb is used as metadata in the recommendation table.
    """
    bytes_per_record = 64                                      # ~64 B per row
    total_data_mb    = (num_records * bytes_per_record) / (1024 ** 2)
 
    # Shuffle data grows with skew and shrinks when more partitions
    # keep each partition small.
    shuffle_mb = round(
        total_data_mb * (1 + data_skew * 0.5) / max(partitions / 100, 0.5),
        2,
    )
 
    # Rough time model (seconds)
    base_time       = (num_records / 1_000_000) * 3.0          # ~3 s / M rows
    partition_cost  = np.log1p(partitions) / np.log1p(100)     # relative overhead
    skew_penalty    = data_skew * 8.0                          # high skew → slow
    mem_benefit     = np.log2(max(executor_memory_gb, 1)) * 0.5
    cache_benefit   = 2.5 if cache_enabled else 0.0
    core_benefit    = np.log2(max(cores, 1)) * 0.3
 
    exec_sec = max(
        0.5,
        base_time * partition_cost + skew_penalty
        - mem_benefit - cache_benefit - core_benefit,
    )
    return round(exec_sec, 3), shuffle_mb
 
 
# Step 2 – recommendation engine (unchanged from original)
def recommend_config(num_records: int, data_skew: float = 0.1,
                     top_k: int = 3) -> pd.DataFrame:
    """
    Given job characteristics, predict execution time for all candidate
    configs and return the top-K fastest recommendations.
    """
    rows = []
    for cfg in CANDIDATE_CONFIGS:
        _, shuffle_mb = simulate_spark_execution(
            num_records,
            cfg["partitions"],
            cfg["executor_memory_gb"],
            cfg["cores"],
            data_skew,
            bool(cfg["cache_enabled"]),
        )
        rows.append({
            "num_records"       : num_records,
            "partitions"        : cfg["partitions"],
            "executor_memory_gb": cfg["executor_memory_gb"],
            "cores"             : cfg["cores"],
            "data_skew"         : data_skew,
            "cache_enabled"     : cfg["cache_enabled"],
            "shuffle_size_mb"   : shuffle_mb,
        })
 
    df_cands = pd.DataFrame(rows)
    df_cands["predicted_exec_sec"] = best_model.predict(
        df_cands[FEATURES].values
    )
    return (
        df_cands
        .sort_values("predicted_exec_sec")
        .head(top_k)
        .reset_index(drop=True)
    )
 
 
# Step 3 – run recommendations for three job profiles
test_cases = [
    (1_000_000,  0.1, "1M records, low skew"),
    (5_000_000,  0.5, "5M records, moderate skew"),
    (10_000_000, 0.8, "10M records, high skew"),
]
 
print("\nAI Recommendation Engine\n")
print("=" * 60)
 
all_recommendations = []
for records, skew, desc in test_cases:
    recs = recommend_config(records, data_skew=skew, top_k=3)
    print(f"\nJob Profile : {desc}")
    print(
        recs[[
            "executor_memory_gb", "cores", "partitions",
            "cache_enabled", "predicted_exec_sec",
        ]].to_string(index=False)
    )
    recs["job_profile"] = desc
    all_recommendations.append(recs)
 
df_recommendations = pd.concat(all_recommendations, ignore_index=True)

## 11. Default vs. AI-Optimized Comparison

In [ ]:
DEFAULT_CONFIG = {
    "partitions"        : 200,   # Spark default
    "executor_memory_gb": 1,
    "cores"             : 1,
    "cache_enabled"     : 0,
}
 
comparison_rows = []
for records, skew, desc in test_cases:
 
    # ── Default config predicted time ──
    _, sh_def = simulate_spark_execution(
        records,
        DEFAULT_CONFIG["partitions"],
        DEFAULT_CONFIG["executor_memory_gb"],
        DEFAULT_CONFIG["cores"],
        skew,
        False,
    )
    default_input = pd.DataFrame([{
        "num_records"       : records,
        "partitions"        : DEFAULT_CONFIG["partitions"],
        "data_skew"         : skew,
        "cache_enabled"     : DEFAULT_CONFIG["cache_enabled"],
    }])
    default_time = best_model.predict(default_input[FEATURES].values)[0]
 
    # ── Best AI-optimised config predicted time ──
    best_rec       = recommend_config(records, skew, top_k=1).iloc[0]
    optimized_time = best_rec["predicted_exec_sec"]
    improvement    = (default_time - optimized_time) / default_time * 100
 
    comparison_rows.append({
        "Job Profile"       : desc,
        "Default Time (s)"  : round(default_time,   1),
        "Optimized Time (s)": round(optimized_time, 1),
        "Improvement (%)"   : round(improvement,    1),
        "Best Memory (GB)"  : int(best_rec["executor_memory_gb"]),
        "Best Cores"        : int(best_rec["cores"]),
        "Best Partitions"   : int(best_rec["partitions"]),
        "Cache"             : "Yes" if best_rec["cache_enabled"] else "No",
    })
 
df_compare = pd.DataFrame(comparison_rows)
print("\nDefault vs AI-Optimized Configuration Comparison\n")
print(df_compare.to_string(index=False))

## 12. Results Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor("#0D1117")
fig.suptitle(
    "AI-Based Spark Job Performance Optimization System\nResults Dashboard",
    fontsize=18, fontweight="bold", color="white", y=1.01,
)
gs   = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.35)
DARK   = "#0D1117"; CARD = "#161B22"; TEXT = "white"
BLUE   = "#2196F3"; GREEN = "#4CAF50"; ORANGE = "#FF9800"; RED = "#F44336"
 
# Plot 1 – Model R² comparison
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor(CARD)
model_names = list(results_ml.keys())
r2_vals     = [results_ml[m]["R2"] for m in model_names]
bar_colors  = [BLUE, GREEN, ORANGE, RED, "#9C27B0", "#00BCD4"]
bars = ax1.bar(model_names, r2_vals,
               color=bar_colors[:len(model_names)], edgecolor="none")
ax1.set_title("Model R² Comparison", color=TEXT, fontweight="bold")
ax1.set_ylim(0.8, 1.0)
ax1.tick_params(colors=TEXT)
ax1.set_facecolor(CARD)
for bar, val in zip(bars, r2_vals):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.002,
        f"{val:.4f}", ha="center", va="bottom", color=TEXT, fontsize=9,
    )
ax1.set_xticklabels(model_names, rotation=10, ha="right")
 
# Plot 2 – Actual vs Predicted
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor(CARD)
preds_best = best_model.predict(X_test)
ax2.scatter(y_test, preds_best, alpha=0.4, s=12, color=BLUE)
lims = [
    min(y_test.min(), preds_best.min()),
    max(y_test.max(), preds_best.max()),
]
ax2.plot(lims, lims, "r--", linewidth=1.5)
ax2.set_title(f"Actual vs Predicted\n({best_name})", color=TEXT, fontweight="bold")
ax2.set_xlabel("Actual (s)", color=TEXT)
ax2.set_ylabel("Predicted (s)", color=TEXT)
ax2.tick_params(colors=TEXT)
 
# Plot 3 – Default vs Optimised bar chart
ax3 = fig.add_subplot(gs[0, 2])
ax3.set_facecolor(CARD)
x     = np.arange(len(df_compare))
width = 0.35
ax3.bar(x - width / 2, df_compare["Default Time (s)"],   width,
        label="Default",   color=RED,   edgecolor="none")
ax3.bar(x + width / 2, df_compare["Optimized Time (s)"], width,
        label="Optimized", color=GREEN, edgecolor="none")
ax3.set_title("Default vs AI-Optimized", color=TEXT, fontweight="bold")
ax3.set_ylabel("Time (s)", color=TEXT)
ax3.set_xticks(x)
ax3.set_xticklabels(
    [r.split(",")[0] for r in df_compare["Job Profile"]],
    rotation=12, ha="right",
)
ax3.tick_params(colors=TEXT)
ax3.legend(facecolor=CARD, labelcolor=TEXT, fontsize=8)
 
# Plot 4 – Improvement %
ax4 = fig.add_subplot(gs[1, 0])
ax4.set_facecolor(CARD)
colors_imp = [GREEN if v > 0 else RED for v in df_compare["Improvement (%)"]]
ax4.bar(range(len(df_compare)), df_compare["Improvement (%)"],
        color=colors_imp, edgecolor="none")
ax4.set_title("Performance Improvement (%)", color=TEXT, fontweight="bold")
ax4.set_xticks(range(len(df_compare)))
ax4.set_xticklabels(
    [r.split(",")[0] for r in df_compare["Job Profile"]],
    rotation=12, ha="right",
)
ax4.tick_params(colors=TEXT)
for i, v in enumerate(df_compare["Improvement (%)"]):
    ax4.text(i, v + 0.5, f"{v:.1f}%", ha="center",
             color=TEXT, fontsize=10, fontweight="bold")
 
# Plot 5 – Live benchmark times
ax5 = fig.add_subplot(gs[1, 1])
ax5.set_facecolor(CARD)
bench_colors = [
    BLUE   if "balanced" in r else
    (GREEN if "many"     in r else ORANGE)
    for r in df_bench["label"]
]
ax5.barh(df_bench["label"], df_bench["time_sec"],
         color=bench_colors, edgecolor="none")
ax5.set_title("Live Spark Benchmark Times", color=TEXT, fontweight="bold")
ax5.set_xlabel("Wall-Clock Time (s)", color=TEXT)
ax5.tick_params(colors=TEXT)
 
# Plot 6 – Feature importance
ax6 = fig.add_subplot(gs[1, 2])
ax6.set_facecolor(CARD)
fi_sorted  = fi_df.sort_values("Importance")
fi_colors  = [ORANGE if v > 0.15 else BLUE for v in fi_sorted["Importance"]]
ax6.barh(fi_sorted["Feature"], fi_sorted["Importance"],
         color=fi_colors, edgecolor="none")
ax6.set_title(f"Feature Importance ({fi_source})", color=TEXT, fontweight="bold")
ax6.set_xlabel("Importance", color=TEXT)
ax6.tick_params(colors=TEXT)
 
# Plot 7 – Prediction error distribution
ax7 = fig.add_subplot(gs[2, 0])
ax7.set_facecolor(CARD)
errors = preds_best - y_test
ax7.hist(errors, bins=35, color=BLUE, edgecolor="none", alpha=0.85)
ax7.axvline(0, color="red", linewidth=1.5, linestyle="--")
ax7.set_title("Prediction Error Distribution", color=TEXT, fontweight="bold")
ax7.set_xlabel("Error (s)", color=TEXT)
ax7.tick_params(colors=TEXT)
 
# Plot 8 – Exec time by executor memory
# (executor_memory_gb is not in df_raw, so we use df_recommendations instead)
ax8 = fig.add_subplot(gs[2, 1])
ax8.set_facecolor(CARD)
mem_levels = sorted(df_recommendations["executor_memory_gb"].unique())
for i, mem in enumerate(mem_levels):
    data_mem = df_recommendations[
        df_recommendations["executor_memory_gb"] == mem
    ]["predicted_exec_sec"]
    ax8.boxplot(
        data_mem,
        positions=[i],
        widths=0.6,
        patch_artist=True,
        boxprops=dict(facecolor=BLUE, color=TEXT),
        medianprops=dict(color=ORANGE, linewidth=2),
        whiskerprops=dict(color=TEXT),
        capprops=dict(color=TEXT),
        flierprops=dict(marker="o", color=TEXT, alpha=0.3, markersize=3),
    )
ax8.set_xticks(range(len(mem_levels)))
ax8.set_xticklabels([f"{m}GB" for m in mem_levels])
ax8.set_title("Predicted Exec Time by Executor Memory",
              color=TEXT, fontweight="bold")
ax8.set_xlabel("Executor Memory", color=TEXT)
ax8.set_ylabel("Predicted Exec Time (s)", color=TEXT)
ax8.tick_params(colors=TEXT)
 
# Plot 9 – Top config recommendation table
ax9 = fig.add_subplot(gs[2, 2])
ax9.set_facecolor(CARD)
ax9.axis("off")
top_recs   = df_recommendations.groupby("job_profile").first().reset_index()
table_data = [["Job Profile", "Mem(GB)", "Cores", "Parts", "Pred(s)"]]
for _, r in top_recs.iterrows():
    table_data.append([
        r["job_profile"].split(",")[0],
        int(r["executor_memory_gb"]),
        int(r["cores"]),
        int(r["partitions"]),
        f"{r['predicted_exec_sec']:.1f}",
    ])
tbl = ax9.table(
    cellText=table_data[1:],
    colLabels=table_data[0],
    loc="center",
    cellLoc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
for (row, col), cell in tbl.get_celld().items():
    cell.set_facecolor("#1F2937" if row == 0 else CARD)
    cell.set_text_props(color=TEXT)
    cell.set_edgecolor("#374151")
ax9.set_title("Top AI Recommendations", color=TEXT, fontweight="bold", pad=15)
 
# Tidy spines for all axes
for ax in fig.get_axes():
    ax.spines["bottom"].set_color("#374151")
    ax.spines["left"].set_color("#374151")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
 
plt.savefig("dashboard.png", dpi=150, bbox_inches="tight", facecolor=DARK)
plt.show()

## 13. Summary

In [ ]:
print("\n" + "=" * 65)
print("  AI-BASED SPARK JOB PERFORMANCE OPTIMIZATION – SUMMARY")
print("=" * 65)
print(f"\n  Dataset       : {len(df_raw):,} simulated job executions")
print(f"  Features used : {FEATURES}")
print(f"\n  ── Model Performance ({best_name}) ──")
print(f"  R² Score      : {results_ml[best_name]['R2']:.4f}")
print(f"  MAE           : {results_ml[best_name]['MAE']:.2f} seconds")
print(f"  RMSE          : {results_ml[best_name]['RMSE']:.2f} seconds")
print(
    f"  CV R² (5-fold): {results_ml[best_name]['CV_mean']:.4f}"
    f" ± {results_ml[best_name]['CV_std']:.4f}"
)
 
print("\n  ── Performance Improvements vs Default Config ──")
for _, row in df_compare.iterrows():
    print(
        f"  {row['Job Profile']:<30} "
        f"↓ {row['Improvement (%)']:>5.1f}% "
        f"({row['Default Time (s)']:.0f}s → {row['Optimized Time (s)']:.0f}s)"
    )
 
print("\n  ── Top Config Drivers ──")
top_feat = fi_df.sort_values("Importance", ascending=False).head(3)
for _, row in top_feat.iterrows():
    print(f"  {row['Feature']:<22} importance = {row['Importance']:.4f}")